# M1 Scale — EDA Inspection Notebook

**Purpose:** Interactive inspection of a pre-generated M1 quality report artifact.

**Rules:**
- Zero network requests.
- Does not crawl, does not modify raw/canonical/derived artifacts.
- Does not write new research results by default.
- Reads one **explicit** quality artifact directory (configure in Cell 1).
- Business logic lives in `src/delta_t1/`, not here.

**Source of truth:** `report.json`, `per_symbol.jsonl`, and `plots/` inside the artifact.

> **Warning — Research gate remains PARTIAL.**  
> Historical identity, financial PIT, and research sample-size policy are **unresolved**.  
> `market_feature_stage_ready = True` only means canonical promotion succeeded and the  
> market feature artifact was generated. It does **not** open M2 or approve clustering.

In [ ]:
# ============================================================
# Cell 1: CONFIGURE — edit this path before running
# ============================================================
from pathlib import Path

# Set to the explicit quality report directory you want to inspect.
# Do NOT use 'latest' — reproducibility requires an explicit artifact ID.
QUALITY_REPORT_DIR = Path(
    "../../data/derived/m1_scale_quality/m1-scale-quality-20260918T152429Z-dcbd8e8e"
)
# Fallback if running from repository root rather than notebooks/eda/
if not QUALITY_REPORT_DIR.exists() and Path("data/derived/m1_scale_quality/m1-scale-quality-20260918T152429Z-dcbd8e8e").exists():
    QUALITY_REPORT_DIR = Path("data/derived/m1_scale_quality/m1-scale-quality-20260918T152429Z-dcbd8e8e")

assert QUALITY_REPORT_DIR.exists(), (
    f"Quality report directory not found: {QUALITY_REPORT_DIR.resolve()}\n"
    "Run scripts/report_m1_scale_quality.py first to generate the artifact."
)
print(f"Using quality report: {QUALITY_REPORT_DIR.name}")

In [ ]:
# ============================================================
# Cell 2: Load report.json
# ============================================================
import json

with open(QUALITY_REPORT_DIR / "report.json", encoding="utf-8") as f:
    report = json.load(f)

print(f"report_id        : {report['report_id']}")
print(f"generated_at     : {report['generated_at']}")
print(f"canonical_run_id : {report['canonical_run_id']}")
print(f"m1_status        : {report['m1_status']}")
print(f"m1_gate_status   : {report['m1_gate_status']}")
print(f"network_requests : {report['network_requests']}")

In [ ]:
# ============================================================
# Cell 3: Readiness summary
# ============================================================
readiness = {
    'market_feature_stage_ready': report.get('market_feature_stage_ready'),
    'feature_stage_ready (deprecated alias)': report.get('feature_stage_ready'),
    'research_stage_ready': report.get('research_stage_ready'),
}
print('Readiness:')
for k, v in readiness.items():
    mark = 'YES' if v else 'NO'
    print(f'  {mark:3s}  {k}: {v}')

print('\nBlocking reasons (research gate):')
for reason in report.get('blocking_reasons', []):
    print(f'  - {reason}')

print('\nWarning: Historical identity and financial PIT remain UNRESOLVED.')
print('  market_feature_stage_ready does NOT open M2 or approve research.')

In [ ]:
# ============================================================
# Cell 4: Summary metrics
# ============================================================
import pandas as pd

# Fallback for display() if running outside interactive IPython/Jupyter
if 'display' not in globals():
    display = print

summary = report['summary']
df_summary = pd.DataFrame([{'Metric': k, 'Value': v} for k, v in summary.items()])
display(df_summary.set_index('Metric'))

In [ ]:
# ============================================================
# Cell 5: Required feature availability
# ============================================================
feat_cov = report['latest_feature_coverage']
df_feat = pd.DataFrame([
    {'Feature': k, 'Available': v['available'], 'Missing': v['missing']}
    for k, v in feat_cov.items()
])
df_feat['Available%'] = (df_feat['Available'] / summary['selected_symbols'] * 100).round(1)
display(df_feat.set_index('Feature').sort_values('Missing', ascending=False))

In [ ]:
# ============================================================
# Cell 6: Exchange coverage
# ============================================================
ex_cov = report['exchange_coverage']
df_ex = pd.DataFrame([
    {'Exchange': k, 'Securities': v['securities'], 'Price rows': v['price_rows']}
    for k, v in sorted(ex_cov.items())
])
display(df_ex.set_index('Exchange'))

In [ ]:
# ============================================================
# Cell 7: Display pre-generated EDA charts
# NOTE: session coverage denominator = observed canonical exchange-session union,
#       NOT a verified official HOSE/HNX/UPCOM exchange calendar.
# ============================================================
try:
    from IPython.display import Image, display as ipydisplay
except ImportError:
    Image = None
    ipydisplay = None

charts = [
    ('exchange_distribution.png',     '1. Securities by Exchange'),
    ('observed_session_coverage.png', '2. Coverage vs. Observed Exchange Sessions'),
    ('feature_availability.png',      '3. Required Feature Availability'),
    ('exclusion_reasons.png',         '4. Top Exclusion / Missing-Feature Reasons'),
]
plots_dir = QUALITY_REPORT_DIR / 'plots'
for fname, caption in charts:
    path = plots_dir / fname
    if path.exists():
        print(f'\n--- {caption} ---')
        if Image and ipydisplay:
            ipydisplay(Image(filename=str(path)))
        else:
            print(f'[{path.name}] ({path.stat().st_size:,} bytes) - rendered in Jupyter notebook')
    else:
        print(f'[missing] {caption}: run pip install -e .[research] and regenerate')

In [ ]:
# ============================================================
# Cell 8: Load per_symbol.jsonl
# ============================================================
rows = []
with open(QUALITY_REPORT_DIR / 'per_symbol.jsonl', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} symbols')
print(f'Columns: {list(df.columns)}')

In [ ]:
# ============================================================
# Cell 9: Highest session coverage vs. observed exchange sessions (top 10)
# Denominator = observed canonical exchange-session union
# (NOT a verified official exchange calendar)
# ============================================================
cols = ['ticker', 'exchange', 'observed_first_date', 'observed_last_date',
        'price_rows', 'observed_session_coverage', 'market_feature_ready']
print('=== Highest coverage vs. observed exchange sessions (top 10) ===')
display(
    df[cols].dropna(subset=['observed_session_coverage'])
      .sort_values('observed_session_coverage', ascending=False)
      .head(10).reset_index(drop=True)
)

In [ ]:
# ============================================================
# Cell 10: Lowest session coverage (bottom 10)
# ============================================================
print('=== Lowest coverage vs. observed exchange sessions (bottom 10) ===')
display(
    df[cols].dropna(subset=['observed_session_coverage'])
      .sort_values('observed_session_coverage', ascending=True)
      .head(10).reset_index(drop=True)
)

In [ ]:
# ============================================================
# Cell 11: market_feature_ready symbols
# ============================================================
mfr = df[df['market_feature_ready'] == True].copy()
print(f'market_feature_ready: {len(mfr)} / {len(df)} symbols')
display(
    mfr[['ticker', 'exchange', 'observed_first_date', 'observed_last_date',
         'price_rows', 'observed_session_coverage']]
      .sort_values('ticker').head(20).reset_index(drop=True)
)

In [ ]:
# ============================================================
# Cell 12: Symbols missing mom_252
# ============================================================
missing_mom252 = df[
    df['missing_required_features'].apply(
        lambda x: 'mom_252' in x if isinstance(x, list) else False
    )
].copy()
print(f'Symbols missing mom_252: {len(missing_mom252)} / {len(df)}')
display(
    missing_mom252[['ticker', 'exchange', 'observed_first_date', 'observed_last_date',
                    'price_rows', 'observed_session_coverage', 'market_feature_ready']]
      .sort_values('price_rows', ascending=False).head(20).reset_index(drop=True)
)

In [ ]:
# ============================================================
# Cell 13: Top exclusion reasons
# ============================================================
df_reasons = pd.DataFrame([
    {'Reason': k, 'Count': v}
    for k, v in report.get('exclusion_reason_counts', {}).items()
]).sort_values('Count', ascending=False).head(20)
display(df_reasons.set_index('Reason'))

print('\nNote: historical_identity:provisional_observed_interval_only is expected.')
print('  Identity is observed-interval only, NOT complete historical membership.')
print('  This is NOT a market-feature calculation error.')

In [ ]:
# ============================================================
# Cell 14: Final readiness status reminder
# ============================================================
print('=' * 60)
print('FINAL READINESS STATUS')
print('=' * 60)
print(f"  market_feature_stage_ready : {report.get('market_feature_stage_ready')}")
print(f"  research_stage_ready       : {report.get('research_stage_ready')}")
print(f"  feature_stage_ready (dep.) : {report.get('feature_stage_ready')}")
print('')
print('Unresolved blockers (research gate remains FAIL/PARTIAL):')
for b in report.get('blocking_reasons', []):
    print(f'  x {b}')
print('')
print('Do NOT proceed to M2 until all three blockers are resolved.')